# Button action

This notebook shows a `Button` that sends an action back to Python. The button event has the name `button_clicked`.

Run the surface cell, then interact with the form. The callback updates the status after the click. Run the last cell to inspect the action.

In [4]:
import sys
from importlib.util import find_spec

# Pyodide and xeus-python both report sys.platform "emscripten"; only Pyodide ships the pyodide package.
if sys.platform != "emscripten":
    KERNEL = "cpython"
else:
    try:
        has_pyodide = find_spec("pyodide") is not None
    except ValueError:  # xeus-python's pyodide stub has __spec__ set to None
        has_pyodide = False
    KERNEL = "pyodide" if has_pyodide else "xpython"

if KERNEL == "pyodide" and find_spec("a2ui_jupyter") is None:
    import piplite

    await piplite.install(["rpds-py", "a2ui-jupyter"])

from a2ui_jupyter import A2UISurface

BASIC_CATALOG_ID = "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json"


def new_surface(surface_id, *, send_data_model=False, **widget_kwargs):
    surface = A2UISurface(**widget_kwargs)
    create = {"surfaceId": surface_id, "catalogId": BASIC_CATALOG_ID}
    if send_data_model:
        create["sendDataModel"] = True
    surface.send({"version": "1.0", "createSurface": create})
    return surface

In [5]:
surface = new_surface("button-action")
surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "button-action",
                "path": "/status",
                "value": "Waiting for a click",
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "button-action",
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["title", "status", "action_button"],
                        "justify": "center",
                        "align": "center",
                    },
                    {
                        "id": "title",
                        "component": "Text",
                        "text": "Click the button below",
                        "variant": "body",
                    },
                    {
                        "id": "status",
                        "component": "Text",
                        "text": {"@path": "/status"},
                        "variant": "caption",
                    },
                    {
                        "id": "action_button",
                        "component": "Button",
                        "child": "button_label",
                        "variant": "primary",
                        "action": {"event": {"name": "button_clicked", "context": {}}},
                    },
                    {"id": "button_label", "component": "Text", "text": "Click me"},
                ],
            },
        },
    ]
)

action_data = None


def handle_action(action):
    global action_data
    action_data = action.model_dump(by_alias=True, exclude_unset=True)
    surface.send(
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "button-action",
                "path": "/status",
                "value": f"Received {action_data['action']['name']} from {action_data['action']['sourceComponentId']}",
            },
        }
    )


surface.on_action(handle_action)
surface

After the `button_clicked` action, inspect its payload:

In [6]:
action_data

{'version': '1.0',
 'action': {'name': 'button_clicked',
  'surfaceId': 'button-action',
  'sourceComponentId': 'action_button',
  'timestamp': '2026-11-21T08:41:05.228Z',
  'context': {}}}